# Bragg peaks: tracked (120) → (111)

Pseudo-Voigt lineouts of the **XPCS-tracked** peak: the file labeled (120) through 350 K, then (111) from 360 K.

- **Panel a** shows $I(|Q|)$ at 320, 339, 360, 380, 399, and 420 K, colored with the locked turbo scale from `temperature_color`.
- **Panel b** tracks $|Q|_0$ and the Scherrer length $\xi$ from a single-peak Pseudo-Voigt at **all nine** temperatures, using each temperature's native tracked ROI (no padding into a neighbor).
- A two-peak Pseudo-Voigt on that same ROI gives the integrated intensity in a high-$Q$ (120-like) window vs a low-$Q$ (111-like) window. That is a lineout decomposition, not a re-indexing of the four 350 K peaks.

Lineouts are a **column sum** of the mean cropped image (`sum` over the horizontal detector axis), plotted vs $Q_y$ from the vertical pixel row. That is not a radial $|Q|$ integral. These peaks sit near $Q_x\approx 0$, so $Q_y\approx|Q|$.

350 K and 370 K have no processed bright-spot crop, so those lineouts are taken from the raw scan cropped to the XPCS ROI of the tracked file.

In [ ]:
import sys
sys.path.append('../src/')

from pathlib import Path
import json

import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.colors import LinearSegmentedColormap
from scipy.ndimage import median_filter
from scipy.optimize import brentq, curve_fit, minimize

try:
    import hdf5plugin  # noqa: F401
except ImportError:
    pass

from postprocess import (
    TRACKED_SCANS,
    temperature_cmap,
    temperature_color,
    temperature_norm,
)
from xpcs import pseudo_voigt


In [ ]:
def apply_paper_style():
    helvetica = font_manager.FontProperties(family='Helvetica')
    font_manager.findfont(helvetica, fallback_to_default=False)
    plt.rcParams.update({
        'font.family': 'sans-serif',
        'font.sans-serif': ['Helvetica', 'Arial'],
        'font.size': 9,
        'axes.labelsize': 11,
        'axes.titlesize': 11,
        'xtick.labelsize': 9,
        'ytick.labelsize': 9,
        'legend.fontsize': 8,
        'legend.frameon': False,
        'axes.linewidth': 1.0,
        'lines.linewidth': 1.2,
        'xtick.direction': 'in',
        'ytick.direction': 'in',
        'xtick.major.width': 1.0,
        'ytick.major.width': 1.0,
        'xtick.major.size': 4.5,
        'ytick.major.size': 4.5,
        'xtick.top': True,
        'ytick.right': True,
        'mathtext.fontset': 'custom',
        'mathtext.rm': 'Helvetica',
        'mathtext.it': 'Helvetica:italic',
        'mathtext.bf': 'Helvetica:bold',
        'pdf.fonttype': 42,
        'ps.fonttype': 42,
        'savefig.dpi': 600,
        'savefig.bbox': 'tight',
        'savefig.pad_inches': 0.03,
    })


apply_paper_style()

cmap = temperature_cmap()
norm = temperature_norm()
FIGSIZE = (11.0, 3.85)
red = '#e31a1c'
blue = '#263fc2'
SPT = (360.0, 373.0)
spt_cmap = LinearSegmentedColormap.from_list(
    'spt_warm',
    ['#fff9b0', '#ffd166', '#ff9f1c', '#e85d04'],
    N=256,
)

DISPLAY_TEMPS = (320, 339, 360, 380, 399, 420)
L_FILE_MM = 2200.0
L_TRUE_MM = 2020.0
Q_REF_120 = 1.1429
SCHERRER_K = 0.9
NFRAMES_PROCESSED = 200
NFRAMES_RAW = 50

Q_120_WIN = (1.125, 1.155)
Q_111_WIN = (1.088, 1.122)

config_path = Path('../configs/config_B10.json')
config = json.loads(config_path.read_text()) if config_path.exists() else {}
APS_BASE = Path(config['Base']) if config.get('Base') else Path('/Users/eriklamb/data/APS/8_ID_E/Na2B10H10')
DESK_PROCESSED = Path('/Users/eriklamb/Desktop/Na2b10h10/Processed')

print('locked temperature colors (display temps)')
for T in DISPLAY_TEMPS:
    rgba = temperature_color(T, cmap=cmap, norm=norm)
    print(f'  {T:g} K  {rgba}')


In [ ]:
def qy_vertical_rows(rows, ver0, l_mm, wavelength, two_theta, pixel_mm):
    ver_angle = two_theta - np.degrees(np.arctan((rows - ver0) * pixel_mm / l_mm))
    return (4.0 * np.pi / wavelength) * np.sin(np.radians(ver_angle / 2.0))


def clean_mean_image(det, gap_frac=0.5, hot_sigma=5):
    med_time = np.median(det, axis=0)
    local_med = median_filter(med_time, size=7)
    local_mad = median_filter(np.abs(med_time - local_med), size=7)
    sigma = np.maximum(1.4826 * local_mad, 1.0)
    gap_mask = (med_time < gap_frac * local_med) & (local_med > 0)
    hot_mask = med_time > local_med + hot_sigma * sigma
    valid = ~(gap_mask | hot_mask)
    return det.mean(axis=0) * valid, valid


def infer_crop_v0(qy_ref, nrows, wavelength, two_theta, pixel_mm, ver0):
    """Detector-row origin of a processed crop, holding Y0 fixed (avoids Qy degeneracy)."""

    def residual(params):
        rows = params[0] + np.arange(nrows)
        q_calc = qy_vertical_rows(rows, ver0, L_FILE_MM, wavelength, two_theta, pixel_mm)
        return np.mean((q_calc - qy_ref) ** 2)

    best = None
    for x0 in (0, 200, 400, 600, 700, 800, 1000):
        result = minimize(
            residual, [float(x0)], method='Nelder-Mead',
            options={'xatol': 1e-4, 'fatol': 1e-16, 'maxiter': 2000},
        )
        if best is None or result.fun < best.fun:
            best = result
    return float(best.x[0])


def find_processed(T, scan):
    desk = sorted(DESK_PROCESSED.glob(f'particleA7_temp{T}K_scan{scan}_*.h5'))
    if desk:
        return desk[0]
    aps = APS_BASE / f'A{scan}_NaBH_att000020_{T}K_001' / 'processed'
    if not aps.exists():
        return None
    hits = sorted(aps.glob(f'particleA7_temp{T}K_scan{scan}_*.h5'))
    hits.sort(key=lambda p: (('NewMask' not in p.name), -p.stat().st_size))
    return hits[0] if hits else None


def find_result(T, scan, hkl):
    folder = APS_BASE / f'A{scan}_NaBH_att000020_{T}K_001' / 'results'
    hits = list(folder.glob(f'*({hkl})*.h5'))
    if not hits:
        raise FileNotFoundError(f'no ({hkl}) result for T={T} scan={scan}')
    return hits[0]


def _scalar(value):
    return float(np.asarray(value).reshape(-1)[0])


def load_processed(path, nframes=NFRAMES_PROCESSED):
    with h5py.File(path, 'r') as f:
        qy_file = np.asarray(f['pre_processing/Qy'][:, 0], dtype=float)
        ntot = f['data/det_corr'].shape[0]
        det = np.asarray(f['data/det_corr'][:min(nframes, ntot)], dtype=float)
        temp = _scalar(f['experimental_parameters/temperature'][()])
        wl = _scalar(f['experimental_parameters/wavelength'][()])
        tth = _scalar(f['experimental_parameters/ttheta'][()])
        pix = _scalar(f['experimental_parameters/pixel_size'][()])
        Y0 = _scalar(f['experimental_parameters/Y0'][()])
    intensity = clean_mean_image(det)[0].sum(axis=1)
    crop_v0 = infer_crop_v0(qy_file, len(intensity), wl, tth, pix, Y0)
    return {
        'temperature': temp, 'wl': wl, 'tth': tth, 'pix': pix, 'Y0': Y0,
        'crop_v0': crop_v0, 'intensity': intensity, 'path': Path(path), 'source': 'processed',
    }


def load_raw_roi(T, scan, hkl, nframes=NFRAMES_RAW, row_pad=(0, 0)):
    result_path = find_result(T, scan, hkl)
    with h5py.File(result_path, 'r') as f:
        roi = np.asarray(f['peaks/peak_0/roi'][()]).astype(int)
        wl = _scalar(f['experimental_parameters/wavelength'][()])
        tth = _scalar(f['experimental_parameters/ttheta'][()])
        pix = _scalar(f['experimental_parameters/pixel_size'][()])
        Y0 = _scalar(f['experimental_parameters/Y0'][()])
    r0, r1, c0, c1 = roi.tolist()
    r0 = max(0, r0 - row_pad[0])
    r1 = r1 + row_pad[1]
    raw = next((APS_BASE / f'A{scan}_NaBH_att000020_{T}K_001').glob('*_001.h5'))
    with h5py.File(raw, 'r') as f:
        det = np.asarray(f['entry/data/data'][:nframes, r0:r1, c0:c1], dtype=float)
    intensity = clean_mean_image(det)[0].sum(axis=1)
    return {
        'temperature': float(T), 'wl': wl, 'tth': tth, 'pix': pix, 'Y0': Y0,
        'crop_v0': float(r0), 'intensity': intensity, 'path': raw, 'source': 'raw',
        'roi': (r0, r1, c0, c1),
    }


In [ ]:
def locate_shoulder_main(q, y):
    i_main = int(np.argmax(y))
    low = y.copy()
    low[(q >= q[i_main] - 0.002) | (q < q[i_main] - 0.03)] = 0
    i_sh = int(np.argmax(low)) if low.max() > 0 else max(0, i_main - 8)
    if q[i_sh] >= q[i_main]:
        i_sh = max(0, i_main - 8)
    return i_sh, i_main


def fit_brightest(q, intensity, shoulder_mode=False):
    """Single Pseudo-Voigt of the brightest peak (original results_compile logic)."""
    y = intensity / intensity.max()
    i_sh, i_main = locate_shoulder_main(q, y)
    c_main = float(q[i_main])
    if shoulder_mode:
        c_sh = float(q[i_sh])
        mask = (q > c_sh - 0.03) & (q < c_main + 0.015) & np.isfinite(y)
    else:
        mask = (q > c_main - 0.06) & (q < c_main + 0.06) & np.isfinite(y)
    q_fit, y_fit = q[mask], y[mask]
    p0 = [max(y_fit.max() - y_fit.min(), 0.05), c_main, 0.01, 0.5, float(np.percentile(y_fit, 10))]
    if shoulder_mode:
        sigma = np.ones_like(y_fit) * 0.15
        sigma[q_fit < c_main - 0.001] = 0.04
        bounds = ([0.0, c_main - 0.015, 1e-4, 0.0, 0.0], [2.0, c_main + 0.015, 0.03, 1.0, 0.12])
        popt, pcov = curve_fit(
            pseudo_voigt, q_fit, y_fit, p0=p0, bounds=bounds,
            sigma=sigma, absolute_sigma=True, maxfev=20000,
        )
    else:
        bounds = ([0.0, c_main - 0.05, 1e-4, 0.0, -np.inf], [3.0, c_main + 0.05, 0.2, 1.0, np.inf])
        popt, pcov = curve_fit(pseudo_voigt, q_fit, y_fit, p0=p0, bounds=bounds, maxfev=20000)
    perr = np.sqrt(np.maximum(np.diag(pcov), 0.0))
    q_fine = np.linspace(q_fit.min(), q_fit.max(), 400)
    y_full = pseudo_voigt(q_fine, *popt)
    y_peak = y_full - popt[4]
    peak_h = float(np.max(y_peak))
    xi_nm = 2.0 * np.pi * SCHERRER_K / popt[2] * 0.1
    xi_err = abs(2.0 * np.pi * SCHERRER_K / popt[2] ** 2 * perr[2]) * 0.1
    return {
        'q_data': q_fit,
        'y_data': (y_fit - popt[4]) / peak_h,
        'q_fine': q_fine,
        'y_fit': y_peak / peak_h,
        'center': float(popt[1]),
        'center_err': float(perr[1]),
        'fwhm': float(popt[2]),
        'fwhm_err': float(perr[2]),
        'xi_nm': float(xi_nm),
        'xi_err_nm': float(xi_err),
        'amplitude': float(popt[0]),
        'baseline': float(popt[4]),
        'shoulder_mode': shoulder_mode,
    }


def two_peak_model(x, a120, c120, w120, e120, a111, c111, w111, e111, bkg):
    return (
        pseudo_voigt(x, a120, c120, w120, e120, 0.0)
        + pseudo_voigt(x, a111, c111, w111, e111, 0.0)
        + bkg
    )


def pick_window_center(q, y, lo, hi, target, min_height=0.12):
    mask = (q >= lo) & (q <= hi) & np.isfinite(y)
    if not np.any(mask):
        return None
    qq, yy = q[mask], y[mask]
    candidates = []
    pad = np.r_[yy[0], yy, yy[-1]]
    for i, (qi, yi) in enumerate(zip(qq, yy)):
        if yi >= pad[i] and yi >= pad[i + 2] and yi >= min_height:
            candidates.append((float(qi), float(yi)))
    if not candidates:
        i = int(np.argmax(yy))
        return float(qq[i]) if yy[i] >= min_height else None
    candidates.sort(key=lambda p: (abs(p[0] - target), -p[1]))
    return candidates[0][0]


def fit_two_peaks(q, intensity):
    """Two area-normalized Pseudo-Voigts. Amplitude is integrated intensity."""
    y = intensity.astype(float)
    mask = np.isfinite(y) & (q >= 1.05) & (q <= 1.17)
    qf, yf = q[mask], y[mask]
    yf = yf / yf.max()
    c120 = pick_window_center(qf, yf, *Q_120_WIN, target=Q_REF_120)
    c111 = pick_window_center(qf, yf, *Q_111_WIN, target=1.110)
    bkg = float(np.percentile(yf, 8))
    h120 = float(np.interp(c120, qf, yf) - bkg) if c120 is not None else 0.02
    h111 = float(np.interp(c111, qf, yf) - bkg) if c111 is not None else 0.02
    p0 = [
        max(h120, 0.02) * 0.012, c120 if c120 is not None else 1.143, 0.010, 0.4,
        max(h111, 0.02) * 0.012, c111 if c111 is not None else 1.110, 0.010, 0.4,
        max(bkg, 0.0),
    ]
    bounds = (
        [0.0, Q_120_WIN[0], 0.004, 0.0, 0.0, Q_111_WIN[0], 0.004, 0.0, 0.0],
        [0.5, Q_120_WIN[1], 0.035, 1.0, 0.5, Q_111_WIN[1], 0.035, 1.0, 0.5],
    )
    popt, pcov = curve_fit(two_peak_model, qf, yf, p0=p0, bounds=bounds, maxfev=50000)
    a120, a111 = float(popt[0]), float(popt[4])
    cen120, cen111 = float(popt[1]), float(popt[5])

    def keep(amp, center, window, present):
        if not present:
            return 0.0
        lo, hi = window
        if center <= lo + 0.002 or center >= hi - 0.002:
            return 0.0
        return amp

    a120 = keep(a120, cen120, Q_120_WIN, c120 is not None)
    a111 = keep(a111, cen111, Q_111_WIN, c111 is not None)
    if a120 > 0 and a111 > 0 and a120 < 0.06 * a111:
        a120 = 0.0
    if a111 > 0 and a120 > 0 and a111 < 0.06 * a120:
        a111 = 0.0
    tot = a120 + a111
    return {
        'popt': popt,
        'q_fit': qf,
        'y_fit': yf,
        'a120': a120,
        'a111': a111,
        'center_120': cen120,
        'center_111': cen111,
        'frac_120': a120 / tot if tot > 0 else np.nan,
        'frac_111': a111 / tot if tot > 0 else np.nan,
    }


In [ ]:
scans = []
for T, scan, hkl in TRACKED_SCANS:
    processed = find_processed(T, scan)
    if processed is not None:
        rec = load_processed(processed)
    else:
        rec = load_raw_roi(T, scan, hkl)
    rec['hkl'] = hkl
    rec['scan'] = scan
    scans.append(rec)
    print(f"{T:g} K  {rec['source']:10s}  crop_v0={rec['crop_v0']:.1f}  n={len(rec['intensity'])}  {rec['path'].name}")

scans.sort(key=lambda d: d['temperature'])
ref = next(s for s in scans if abs(s['temperature'] - 320) < 1)
peak_global = ref['crop_v0'] + int(np.argmax(ref['intensity']))


def q_at(ver0):
    return qy_vertical_rows(
        np.array([peak_global]), ver0, L_TRUE_MM, ref['wl'], ref['tth'], ref['pix']
    )[0]


ver0_cal = float(brentq(lambda v: q_at(v) - Q_REF_120, 0.0, 2500.0))
print(
    f"Q calibration: L={L_TRUE_MM:.0f} mm, anchor T={ref['temperature']:.0f} K "
    f"(120) at Q={Q_REF_120:.4f} A^-1, ver0={ver0_cal:.1f} px"
)

for rec in scans:
    rows = rec['crop_v0'] + np.arange(len(rec['intensity']))
    rec['q'] = qy_vertical_rows(rows, ver0_cal, L_TRUE_MM, rec['wl'], rec['tth'], rec['pix'])
    T = rec['temperature']
    rec['bright'] = fit_brightest(
        rec['q'], rec['intensity'],
        shoulder_mode=(T <= 330 or abs(T - 380) < 2),
    )
    rec['two'] = fit_two_peaks(rec['q'], rec['intensity'])
    b = rec['bright']
    tw = rec['two']
    print(
        f"T={T:.0f} K  Q0={b['center']:.4f}  FWHM={b['fwhm']:.4f}  xi={b['xi_nm']:.1f} nm"
        f"  f(120)={tw['frac_120']:.3f}  f(111)={tw['frac_111']:.3f}"
    )


In [ ]:
fig, (ax_fit, ax_t) = plt.subplots(1, 2, figsize=FIGSIZE)
display = {int(round(s['temperature'])): s for s in scans}

for i, T in enumerate(DISPLAY_TEMPS):
    rec = display[T]
    color = temperature_color(T, cmap=cmap, norm=norm)
    fit = rec['bright']
    z = i + 1
    ax_fit.fill_between(fit['q_fine'], 0.0, fit['y_fit'], color=color, alpha=0.40, zorder=2 * z, lw=0)
    ax_fit.plot(fit['q_data'], fit['y_data'], color=color, lw=1.6, label=f'{T:g} K', zorder=2 * z + 1)

ax_fit.set_xlabel(r'$|\mathbf{Q}|$ ($\mathrm{\AA}^{-1}$)')
ax_fit.set_ylabel('I (arb.)')
ax_fit.set_ylim(0.0, 1.08)
ax_fit.set_xlim(1.055, 1.165)
ax_fit.legend(loc='upper left', handlelength=1.2, labelspacing=0.25)
ax_fit.spines['top'].set_visible(False)
ax_fit.spines['right'].set_visible(False)
ax_fit.tick_params(top=False, right=False)

temps = np.array([s['temperature'] for s in scans])
q0 = np.array([s['bright']['center'] for s in scans])
q0_e = np.array([s['bright']['center_err'] for s in scans])
xi = np.array([s['bright']['xi_nm'] for s in scans])
xi_e = np.array([s['bright']['xi_err_nm'] for s in scans])

ax_xi = ax_t.twinx()
ax_t.set_xlim(312, 428)
ax_t.set_ylim(1.085, 1.155)
ax_xi.set_ylim(0, 110)
ax_t.imshow(
    np.linspace(0, 1, 512).reshape(1, -1),
    aspect='auto',
    cmap=spt_cmap,
    extent=[SPT[0], SPT[1], *ax_t.get_ylim()],
    origin='lower',
    interpolation='bicubic',
    zorder=0,
)
ax_t.set_xlim(312, 428)

ax_t.errorbar(
    temps, q0, yerr=q0_e, fmt='o-', color=blue, ecolor=blue,
    elinewidth=1.0, capsize=3, capthick=0.9, markersize=6,
    markerfacecolor=blue, markeredgecolor=blue, lw=1.4, zorder=4,
    label=r'$|\mathbf{Q}_0|$',
)
ax_xi.errorbar(
    temps, xi, yerr=xi_e, fmt='s--', color=red, ecolor=red,
    elinewidth=1.0, capsize=3, capthick=0.9, markersize=6,
    markerfacecolor=red, markeredgecolor=red, lw=1.4, zorder=4,
    label=r'$\xi$',
)

ax_t.set_xlabel('T (K)')
ax_t.set_ylabel(r'$|\mathbf{Q}_0|$ ($\mathrm{\AA}^{-1}$)', color=blue)
ax_xi.set_ylabel(r'$\xi$ (nm)', color=red)
ax_t.set_xticks([320, 340, 360, 380, 400, 420])
ax_t.tick_params(axis='y', colors=blue)
ax_xi.tick_params(axis='y', colors=red)
ax_t.spines['left'].set_color(blue)
ax_xi.spines['right'].set_color(red)
ax_t.spines['right'].set_visible(False)
ax_xi.spines['left'].set_visible(False)

h1, l1 = ax_t.get_legend_handles_labels()
h2, l2 = ax_xi.get_legend_handles_labels()
ax_t.legend(h1 + h2, l1 + l2, loc='lower left')

fig.tight_layout(w_pad=2.0)
out = Path('../figures')
out.mkdir(exist_ok=True)
fig.savefig(out / 'bragg_peaks_na2b10h10.pdf')
fig.savefig(out / 'bragg_peaks_na2b10h10.png')
print('saved', out / 'bragg_peaks_na2b10h10.pdf')


In [ ]:
print(f"{'T (K)':>6}  {'hkl':>5}  {'|Q0|':>10}  {'±':>8}  {'ξ (nm)':>8}  {'±':>8}  {'FWHM':>8}  {'±':>8}")
print('-' * 72)
fit_path = Path('../figures/bragg_peaks_tracked_fits.csv')
fit_path.parent.mkdir(exist_ok=True)
lines = ['T_K,hkl,Q0_Ainv,Q0_err,xi_nm,xi_err,FWHM_Ainv,FWHM_err']
for rec in scans:
    b = rec['bright']
    T = rec['temperature']
    hkl = rec['hkl']
    print(
        f"{T:6.0f}  {hkl:>5}  {b['center']:10.5f}  {b['center_err']:8.5f}  "
        f"{b['xi_nm']:8.2f}  {b['xi_err_nm']:8.2f}  {b['fwhm']:8.5f}  {b['fwhm_err']:8.5f}"
    )
    lines.append(
        f"{T:.0f},{hkl},{b['center']:.6f},{b['center_err']:.6f},"
        f"{b['xi_nm']:.4f},{b['xi_err_nm']:.4f},{b['fwhm']:.6f},{b['fwhm_err']:.6f}"
    )
fit_path.write_text('\n'.join(lines) + '\n')
print('saved', fit_path)
